# 04 - Chunking Deep Dive

## Definition
Chunking splits long documents into retrieval units suitable for embedding and vector search.

## Theory
Chunk size and overlap trade off between context completeness and retrieval precision.

## Motivation
Poor chunking causes context fragmentation and low recall for answer-bearing passages.

## Architecture
Document -> strategy-specific segmentation -> chunk metadata -> vector index.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [ ]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [ ]:
bundle = runner.ingest_dataset()
docs = bundle['documents'][:80]
queries = bundle['queries'][:80]

strategies = [
    ChunkingStrategy.FIXED,
    ChunkingStrategy.RECURSIVE,
    ChunkingStrategy.SEMANTIC,
    ChunkingStrategy.PARENT_CHILD,
]

rows = []
for strategy in strategies:
    out = runner.run_chunking(docs, strategy=strategy)
    runner.index_chunks(out.chunks, reset=True)
    summary, _, _ = runner.evaluator.evaluate_retrieval(queries=queries, top_k=6, max_queries=len(queries))
    rows.append({
        'strategy': strategy.value,
        'num_chunks': len(out.chunks),
        'avg_chunk_chars': out.avg_chunk_length,
        'mrr': summary.mrr,
        'ndcg': summary.ndcg,
    })

pd.DataFrame(rows).sort_values('mrr', ascending=False)


## Tradeoffs
- Fixed: simple and fast, weakest boundary quality
- Recursive: strong practical default
- Semantic: better topical coherence, extra embedding compute
- Parent-child: stronger context recovery, higher storage/index cost